## Business Scenario

You have joined an e-commerce company as a **Senior Data Scientist**. The marketing department wants to build a machine-learning model that predicts whether a customer will make a **repeat purchase**. Before any model can be trained, the raw customer dataset must first be assessed, cleaned, and prepared using appropriate preprocessing techniques.

This notebook documents the complete workflow across seven tasks — from initial exploration through cleaning, transformation, and reduction, to a proximity analysis and a closing reflection.

---

# Task 1 · Dataset Understanding

The goal of this first task is to understand the **structure** of the dataset — its dimensions, attribute names, data types, sample records, and summary statistics — and to separate numerical from categorical attributes. This early exploration reveals what preprocessing the later tasks will need.

In [5]:

# Import Required Libraries


import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances




# Display all columns in the notebook output
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:

# Load Dataset


df = pd.read_csv("mall_customer_dataset.csv")

In [7]:

# Display Dataset Dimensions


rows= df.shape[0]
columns = df.shape[1]

print(f"Number of Records (Rows): {rows}")
print(f"Number of Attributes (Columns): {columns}")

Number of Records (Rows): 2200
Number of Attributes (Columns): 25


In [8]:

# Display Attribute Names


print("List of Attributes:\n")

for column in df.columns:
    print(column)

List of Attributes:

CustomerID
CustomerName
Age
Gender
AnnualIncome_INR
IncomeCurrency
SpendingScore_1_100
MembershipTier
JoinDate
LastPurchaseDate
VisitFrequency
AvgBasketValue_INR
TotalPurchases
OnlinePurchases
StorePurchases
PreferredCategory
City
Country
EmailProvider
DeviceType
PaymentMethod
LoyaltyPoints
SatisfactionRating_1_5
CouponUsed
ChurnNextMonth


In [9]:

# Display First Five Records


df.head(n = 5)

,CustomerID,CustomerName,Age,Gender,AnnualIncome_INR,IncomeCurrency,SpendingScore_1_100,MembershipTier,JoinDate,LastPurchaseDate,VisitFrequency,AvgBasketValue_INR,TotalPurchases,OnlinePurchases,StorePurchases,PreferredCategory,City,Country,EmailProvider,DeviceType,PaymentMethod,LoyaltyPoints,SatisfactionRating_1_5,CouponUsed,ChurnNextMonth
0,CUST100001,Diya Verma,42.0,Female,241279.0,NaN,57,Silver,24-Feb-2024,14/09/2024,RARELY,758.43,40.0,3,38,Beauty,bengaluru,NaN,gmail.com,android,Credit Card,1438.0,2.0,1,0.0
1,CUST100002,Aditya Nair,52.0,male,1135484.0,Rs,61,GOLD,02/02/2025,10-Oct-2025,Weekly,2132.28,7.0,4,2,SPORTS,PUNE,INDIA,hotmail.com,NaN,NaN,865.0,1.0,1,0.0
2,CUST100003,Saanvi Kumar,27.0,M,241860.0,INR,64,Silvr,2022-05-30,2023-12-17,RARELY,2269.85,9.0,0,10,Electronic,lucknow,India,gmail.com,Mobile,UPI,2472.0,1.0,NaN,0.0
3,CUST100004,Vihaan Iyer,27.0,FEMALE,470187.0,INR,86,Silver,01-31-2024,18-Jul-2024,RARELY,723.67,30.0,29,1,Food Court,Pune,NaN,gmail.com,android,Credit Card,1700.0,3.0,NaN,0.0
4,CUST100005,Krishna Rao,25.0,M,630873.0,INR,53,basic,2022-01-25,01-May-2022,Monthly,12062.69,38.0,21,17,Fashion,Bengaluru,India,hotmail.com,Web,CREDIT CARD,1203.0,2.0,1,0.0


In [10]:

# Display Data Types


df.dtypes

CustomerID                 object
CustomerName               object
Age                       float64
Gender                     object
AnnualIncome_INR          float64
IncomeCurrency             object
SpendingScore_1_100         int64
MembershipTier             object
JoinDate                   object
LastPurchaseDate           object
VisitFrequency             object
AvgBasketValue_INR        float64
TotalPurchases            float64
OnlinePurchases             int64
StorePurchases              int64
PreferredCategory          object
City                       object
Country                    object
EmailProvider              object
DeviceType                 object
PaymentMethod              object
LoyaltyPoints             float64
SatisfactionRating_1_5    float64
CouponUsed                 object
ChurnNextMonth            float64
dtype: object

In [11]:

# Information of the Attributes


print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              2200 non-null   object 
 1   CustomerName            2200 non-null   object 
 2   Age                     2096 non-null   float64
 3   Gender                  1970 non-null   object 
 4   AnnualIncome_INR        2059 non-null   float64
 5   IncomeCurrency          1882 non-null   object 
 6   SpendingScore_1_100     2200 non-null   int64  
 7   MembershipTier          1957 non-null   object 
 8   JoinDate                2200 non-null   object 
 9   LastPurchaseDate        2124 non-null   object 
 10  VisitFrequency          1941 non-null   object 
 11  AvgBasketValue_INR      2172 non-null   float64
 12  TotalPurchases          2189 non-null   float64
 13  OnlinePurchases         2200 non-null   int64  
 14  StorePurchases          2200 non-null   

In [15]:

# Descriptive Statistics for Numerical Attributes


df.describe()

,Age,AnnualIncome_INR,SpendingScore_1_100,AvgBasketValue_INR,TotalPurchases,OnlinePurchases,StorePurchases,LoyaltyPoints,SatisfactionRating_1_5,ChurnNextMonth
count,2096.000000,2.059000e+03,2200.00000,2172.000000,2189.000000,2200.000000,2200.000000,2188.000000,2170.000000,2173.000000
mean,34.781966,1.706644e+06,53.90000,13356.393485,90.349018,8.139091,9.426364,8949.128428,3.137327,0.208928
std,18.054272,9.558517e+06,21.44778,104332.718769,850.619369,11.089110,11.598287,87720.316207,1.697376,0.406636
min,-5.000000,-5.000000e+04,1.00000,-250.000000,-3.000000,0.000000,0.000000,-50.000000,0.000000,0.000000
25%,26.000000,4.423870e+05,39.00000,1083.787500,5.000000,1.000000,2.000000,747.750000,2.000000,0.000000
50%,33.000000,6.511680e+05,54.00000,1814.650000,12.000000,4.000000,5.000000,1199.000000,3.000000,0.000000
75%,42.000000,9.591475e+05,68.00000,3044.790000,25.000000,11.000000,13.000000,1618.750000,4.000000,0.000000
max,180.000000,1.000000e+08,100.00000,999999.000000,9999.000000,104.000000,110.000000,999999.000000,11.000000,1.000000


In [14]:

# Descriptive Statistics Including Categorical Attributes


df.describe(include='all')

,CustomerID,CustomerName,Age,Gender,AnnualIncome_INR,IncomeCurrency,SpendingScore_1_100,MembershipTier,JoinDate,LastPurchaseDate,VisitFrequency,AvgBasketValue_INR,TotalPurchases,OnlinePurchases,StorePurchases,PreferredCategory,City,Country,EmailProvider,DeviceType,PaymentMethod,LoyaltyPoints,SatisfactionRating_1_5,CouponUsed,ChurnNextMonth
count,2200,2200,2096.000000,1970,2.059000e+03,1882,2200.00000,1957,2200,2124,1941,2172.000000,2189.000000,2200.000000,2200.000000,2200,2184,1838,1972,1946,1980,2188.000000,2170.000000,1935,2173.000000
unique,2087,556,NaN,9,NaN,4,NaN,9,1830,1781,7,NaN,NaN,NaN,NaN,44,36,5,8,8,8,NaN,NaN,8,NaN
top,CUST100340,Rahul Mehta,NaN,MALE,NaN,INR,NaN,Gold,not available,10-19-2024,Monthly,NaN,NaN,NaN,NaN,Fashion,Chennai,Bharat,gmail.com,IOS,upi,NaN,NaN,yes,NaN
freq,4,12,NaN,240,NaN,962,NaN,247,11,5,302,NaN,NaN,NaN,NaN,69,86,392,277,259,269,NaN,NaN,249,NaN
mean,NaN,NaN,34.781966,NaN,1.706644e+06,NaN,53.90000,NaN,NaN,NaN,NaN,13356.393485,90.349018,8.139091,9.426364,NaN,NaN,NaN,NaN,NaN,NaN,8949.128428,3.137327,NaN,0.208928
std,NaN,NaN,18.054272,NaN,9.558517e+06,NaN,21.44778,NaN,NaN,NaN,NaN,104332.718769,850.619369,11.089110,11.598287,NaN,NaN,NaN,NaN,NaN,NaN,87720.316207,1.697376,NaN,0.406636
min,NaN,NaN,-5.000000,NaN,-5.000000e+04,NaN,1.00000,NaN,NaN,NaN,NaN,-250.000000,-3.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,-50.000000,0.000000,NaN,0.000000
25%,NaN,NaN,26.000000,NaN,4.423870e+05,NaN,39.00000,NaN,NaN,NaN,NaN,1083.787500,5.000000,1.000000,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,747.750000,2.000000,NaN,0.000000
50%,NaN,NaN,33.000000,NaN,6.511680e+05,NaN,54.00000,NaN,NaN,NaN,NaN,1814.650000,12.000000,4.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,1199.000000,3.000000,NaN,0.000000
75%,NaN,NaN,42.000000,NaN,9.591475e+05,NaN,68.00000,NaN,NaN,NaN,NaN,3044.790000,25.000000,11.000000,13.000000,NaN,NaN,NaN,NaN,NaN,NaN,1618.750000,4.000000,NaN,0.000000


In [ ]:

# Identify Numerical and Categorical Attributes


numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("\nNumerical Attributes:")
print(numerical_columns)

print("\nCategorical Attributes:")
print(categorical_columns)


Numerical Attributes:
['Age', 'AnnualIncome_INR', 'SpendingScore_1_100', 'AvgBasketValue_INR', 'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'LoyaltyPoints', 'SatisfactionRating_1_5', 'ChurnNextMonth']

Categorical Attributes:
['CustomerID', 'CustomerName', 'Gender', 'IncomeCurrency', 'MembershipTier', 'JoinDate', 'LastPurchaseDate', 'VisitFrequency', 'PreferredCategory', 'City', 'Country', 'EmailProvider', 'DeviceType', 'PaymentMethod', 'CouponUsed']


## Task 1 · Summary

The dataset was loaded and inspected successfully. It contains **2,200 records** across **25 attributes**, split into **10 numerical** and **15 categorical** columns, with `ChurnNextMonth` as the (imbalanced) binary target.

Even at this stage two issues are visible: `JoinDate` and `LastPurchaseDate` are stored as text rather than dates, and several categorical fields carry mixed spellings and casing. These observations set up the next stage — **Data Quality Assessment** — where missing values, duplicates, inconsistent categories, outliers, and invalid entries are quantified.

# Task 2 · Data Quality Assessment

Before cleaning, the dataset is audited column-by-column to **quantify** its problems: missing values, duplicate records, inconsistent categorical values, outliers (via the IQR rule), invalid entries, and data-type issues. The findings are collected into a single summary report.

In [ ]:

# Data Quality Assessment Report


quality_report = []

for col in df.columns:

    # Basic Information
    data_type = df[col].dtype
    missing = df[col].isnull().sum()
    missing_percent = round((missing / len(df)) * 100, 2)
    unique = df[col].nunique(dropna=True)
    duplicate = df[col].duplicated().sum()

    # Default values
    invalid_entries = 0
    outliers = "N/A"

    # Numerical Columns
    if pd.api.types.is_numeric_dtype(df[col]):

        # Invalid values (negative numbers)
        invalid_entries = (df[col] < 0).sum()

        # IQR Method for Outliers
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    quality_report.append({
        "Column": col,
        "Data Type": str(data_type),
        "Missing Values": missing,
        "Missing %": missing_percent,
        "Unique Values": unique,
        "Duplicate Values": duplicate,
        "Invalid Entries": invalid_entries,
        "Outlier Count": outliers
    })

# Create DataFrame
quality_df = pd.DataFrame(quality_report)

# Display Report

print("DATA QUALITY ASSESSMENT REPORT")

display(quality_df)

# Save Report
quality_df.to_csv("MallCustomer_DataQuality_Report.csv", index=False)

print("\nData Quality Report has been saved successfully as 'MallCustomer_DataQuality_Report.csv'")

DATA QUALITY ASSESSMENT REPORT


,Column,Data Type,Missing Values,Missing %,Unique Values,Duplicate Values,Invalid Entries,Outlier Count
0,CustomerID,object,0,0.00,2087,113,0,N/A
1,CustomerName,object,0,0.00,556,1644,0,N/A
2,Age,float64,104,4.73,74,2125,8,63
3,Gender,object,230,10.45,9,2190,0,N/A
4,AnnualIncome_INR,float64,141,6.41,1964,235,5,87
5,IncomeCurrency,object,318,14.45,4,2195,0,N/A
6,SpendingScore_1_100,int64,0,0.00,98,2102,0,0
7,MembershipTier,object,243,11.05,9,2190,0,N/A
8,JoinDate,object,0,0.00,1830,370,0,N/A
9,LastPurchaseDate,object,76,3.45,1781,418,0,N/A



Data Quality Report has been saved successfully as 'MallCustomer_DataQuality_Report.csv'


## Task 2 · Summary

The audit produced a per-attribute quality report covering data types, missing counts, unique values, duplicates, invalid entries, and outlier counts.

Key findings: the dataset carries **2,824 missing cells (~5%)** spread across 18 columns, **64 full-row duplicates**, widespread **categorical inconsistency** (Gender in 9 variants, City in 36, PreferredCategory in 44), several **invalid negatives** (age, income, basket value), and hundreds of **IQR outliers** in the behavioural counts. Two columns hold dates as plain text.

These results define the cleaning plan applied in Task 3: imputation, duplicate removal, category standardisation, type conversion, and outlier treatment.

# Task 3 · Data Cleaning

## Objective

Each problem identified in Task 2 is now addressed with a targeted preprocessing technique. For every step the notebook shows the **code**, the state **before** and **after**, and the **reason** for the choice.

Steps performed:

- Missing-value imputation
- Duplicate-record removal
- Category standardisation
- Data-type conversion
- Noise handling
- Outlier treatment (IQR capping)

In [ ]:

# 3.1 Missing Value Imputation


print("Missing Values Before Cleaning:\n")
print(df.isnull().sum())

# Fill Numerical Columns with Median
for col in df.select_dtypes(include=['int64','float64']).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill Categorical Columns with Mode
for col in df.select_dtypes(include=['object','category']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing Values After Cleaning:\n")
print(df.isnull().sum())

Missing Values Before Cleaning:

CustomerID                  0
CustomerName                0
Age                       104
Gender                    230
AnnualIncome_INR          141
IncomeCurrency            318
SpendingScore_1_100         0
MembershipTier            243
JoinDate                    0
LastPurchaseDate           76
VisitFrequency            259
AvgBasketValue_INR         28
TotalPurchases             11
OnlinePurchases             0
StorePurchases              0
PreferredCategory           0
City                       16
Country                   362
EmailProvider             228
DeviceType                254
PaymentMethod             220
LoyaltyPoints              12
SatisfactionRating_1_5     30
CouponUsed                265
ChurnNextMonth             27
dtype: int64

Missing Values After Cleaning:

CustomerID                0
CustomerName              0
Age                       0
Gender                    0
AnnualIncome_INR          0
IncomeCurrency            0
Spe

**Reason.** Median imputation is robust to the skew and outliers present in the numeric columns, so it distorts the distribution far less than the mean. Mode imputation fills categorical gaps with the most representative level. Both preserve the ~5% of rows that would otherwise be lost to deletion.

In [ ]:

# 3.2 Duplicate Record Removal


duplicates_before = df.duplicated().sum()

print("Duplicate Records Before:", duplicates_before)

df.drop_duplicates(inplace=True)

duplicates_after = df.duplicated().sum()

print("Duplicate Records After :", duplicates_after)

Duplicate Records Before: 64
Duplicate Records After : 0


**Reason.** Duplicate customer records inflate the influence of repeated observations and can leak between the train and test splits, biasing model evaluation. Removing them cleans 64 rows (2,200 → 2,136).

In [ ]:

# Step 3.3: Category Standardization


# Categorical columns to standardize
categorical_cols = [
    'Gender',
    'IncomeCurrency',
    'MembershipTier',
    'PreferredCategory',
    'City',
    'Country',
    'DeviceType',
    'PaymentMethod',
    'CouponUsed'
]


# Before Standardization

print("="*80)
print("UNIQUE VALUES BEFORE STANDARDIZATION")
print("="*80)

for col in categorical_cols:
    print(f"\n{col}:")
    print(sorted(df[col].astype(str).unique()))


# Category Standardization


# Remove leading/trailing spaces from all categorical columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# Gender
df['Gender'] = df['Gender'].replace({
    'M': 'Male',
    'F': 'Female'
})

# Income Currency
df['IncomeCurrency'] = df['IncomeCurrency'].replace({
    'Inr': 'INR',
    'Rs': 'INR',
    '₹': 'INR'
})

# Membership Tier
df['MembershipTier'] = df['MembershipTier'].replace({
    'Silvr': 'Silver'
})

# Preferred Category
df['PreferredCategory'] = df['PreferredCategory'].replace({
    'Electronic': 'Electronics',
    'Eletronics': 'Electronics',
    'Groceries': 'Grocery',
    'Grocerry': 'Grocery'
})

# City
df['City'] = df['City'].replace({
    'Bangalore': 'Bengaluru',
    'Hyd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    'Chenai': 'Chennai',
    'Mumbay': 'Mumbai',
    'Calcutta': 'Kolkata',
    'New Delhi': 'Delhi'
})

# Country
df['Country'] = df['Country'].replace({
    'Bharat': 'India',
    'In': 'India'
})

# Device Type
df['DeviceType'] = df['DeviceType'].replace({
    'Ios': 'iOS'
})

# Payment Method
df['PaymentMethod'] = df['PaymentMethod'].replace({
    'Card': 'Credit Card',
    'Upi': 'UPI'
})

# Coupon Used
df['CouponUsed'] = df['CouponUsed'].replace({
    '1': 'Yes',
    'Y': 'Yes',
    '0': 'No',
    'N': 'No'
})


# After Standardization

print("\n" + "="*80)
print("UNIQUE VALUES AFTER STANDARDIZATION")


for col in categorical_cols:
    print(f"\n{col}:")
    print(sorted(df[col].unique()))

UNIQUE VALUES BEFORE STANDARDIZATION

Gender:
['F', 'Female', 'M', 'Male', 'Other']

IncomeCurrency:
['Inr', 'Rs', 'Usd', '₹']

MembershipTier:
['Basic', 'Gold', 'Platinum', 'Silver', 'Silvr']

PreferredCategory:
['Beauty', 'Books', 'Electronic', 'Electronics', 'Eletronics', 'Fashion', 'Food Court', 'Gaming', 'Groceries', 'Grocerry', 'Grocery', 'Home Decor', 'Kids', 'Sports']

City:
['Ahmedabad', 'Bangalore', 'Bengaluru', 'Calcutta', 'Chenai', 'Chennai', 'Delhi', 'Hyd', 'Hyderabad', 'Hyderbad', 'Jaipur', 'Kolkata', 'Lucknow', 'Mumbai', 'Mumbay', 'New Delhi', 'Pune']

Country:
['Bharat', 'In', 'India']

DeviceType:
['Android', 'Desktop', 'Ios', 'Mobile', 'Web']

PaymentMethod:
['Card', 'Cash', 'Credit Card', 'Debit Card', 'Upi', 'Wallet']

CouponUsed:
['0', '1', 'N', 'No', 'Y', 'Yes']

UNIQUE VALUES AFTER STANDARDIZATION

Gender:
['Female', 'Male', 'Other']

IncomeCurrency:
['INR', 'Usd']

MembershipTier:
['Basic', 'Gold', 'Platinum', 'Silver']

PreferredCategory:
['Beauty', 'Books', 'E

In [ ]:

# Step 3.4: Data Type Conversion


print("Before Conversion:\n")
print(df[['JoinDate', 'LastPurchaseDate']].dtypes)

# Convert mixed-format dates
df['JoinDate'] = pd.to_datetime(
    df['JoinDate'],
    errors='coerce',
    dayfirst=True
)

df['LastPurchaseDate'] = pd.to_datetime(
    df['LastPurchaseDate'],
    errors='coerce',
    dayfirst=True
)

print("\nAfter Conversion:\n")
print(df[['JoinDate', 'LastPurchaseDate']].dtypes)

# Verify conversion
print("\nMissing values after conversion:")
print(df[['JoinDate', 'LastPurchaseDate']].isnull().sum())

Before Conversion:

JoinDate            datetime64[ns]
LastPurchaseDate    datetime64[ns]
dtype: object

After Conversion:

JoinDate            datetime64[ns]
LastPurchaseDate    datetime64[ns]
dtype: object

Missing values after conversion:
JoinDate            30
LastPurchaseDate     0
dtype: int64


In [ ]:

# Step 3.5: Noise Handling


# Count string columns
text_columns = df.select_dtypes(include='object').columns

# Remove leading and trailing spaces
for col in text_columns:
    df[col] = df[col].str.strip()

print("Noise handling completed successfully.")
print(f"Processed {len(text_columns)} text columns.")

Noise handling completed successfully.
Processed 15 text columns.


In [ ]:

# Step 3.6: Outlier Treatment using IQR Capping

numerical_columns = [
    'Age',
    'AnnualIncome_INR',
    'SpendingScore_1_100',
    'AvgBasketValue_INR',
    'TotalPurchases',
    'OnlinePurchases',
    'StorePurchases',
    'LoyaltyPoints',
    'SatisfactionRating_1_5'
]

outlier_summary = []

for col in numerical_columns:

    # Calculate IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers before treatment
    before = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    # Apply IQR Capping
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    # Count outliers after treatment
    after = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append([col, before, after])

# Summary Table
outlier_report = pd.DataFrame(
    outlier_summary,
    columns=["Column", "Outliers Before", "Outliers After"]
)

print("Outlier Treatment Summary")
display(outlier_report)

Outlier Treatment Summary


,Column,Outliers Before,Outliers After
0,Age,67,0
1,AnnualIncome_INR,107,0
2,SpendingScore_1_100,0,0
3,AvgBasketValue_INR,139,0
4,TotalPurchases,109,0
5,OnlinePurchases,140,0
6,StorePurchases,142,0
7,LoyaltyPoints,17,0
8,SatisfactionRating_1_5,28,0


## Task 3 · Summary

The dataset was cleaned end-to-end. Missing values were imputed (2,824 → 0), 64 duplicate rows were removed (2,200 → 2,136), categorical labels were standardised, the two date columns were converted to `datetime`, residual text noise was stripped, and numeric outliers were treated with **IQR capping**.

The result is a consistent, correctly-typed, gap-free frame of **2,136 records**, ready for transformation in Task 4.

# Task 4 · Data Transformation

## Objective

Transformation reshapes the cleaned data into a form suitable for machine-learning algorithms — placing numerical attributes on comparable scales and converting categorical attributes into numeric representations.

Five transformations are applied and compared against their originals:

1. Min-Max Normalisation
2. Standardisation (Z-score)
3. Log Transformation
4. Label Encoding
5. One-Hot Encoding

In [ ]:

# Step 4.1: Min Max Normalization


# Create a copy of the cleaned dataset
df_transformed = df.copy()

print("Before Min-Max Normalization")
display(df[['AnnualIncome_INR']].head())

# Apply Min-Max Normalization
minmax_scaler = MinMaxScaler()

df_transformed['AnnualIncome_MinMax'] = minmax_scaler.fit_transform(
    df[['AnnualIncome_INR']]
)

print("\nAfter Min-Max Normalization")
display(df_transformed[['AnnualIncome_INR', 'AnnualIncome_MinMax']].head())

Before Min-Max Normalization


,AnnualIncome_INR
0,241279.0
1,1135484.0
2,241860.0
3,470187.0
4,630873.0



After Min-Max Normalization


,AnnualIncome_INR,AnnualIncome_MinMax
0,241279.0,0.173902
1,1135484.0,0.707766
2,241860.0,0.174248
3,470187.0,0.310566
4,630873.0,0.406500


In [ ]:

# Step 4.2: Min Max Normalization


print("Before Standardization")
display(df[['AvgBasketValue_INR']].head())

standard_scaler = StandardScaler()

df_transformed['AvgBasketValue_Standardized'] = standard_scaler.fit_transform(
    df[['AvgBasketValue_INR']]
)

print("\nAfter Standardization")
display(df_transformed[['AvgBasketValue_INR',
                        'AvgBasketValue_Standardized']].head())

Before Standardization


,AvgBasketValue_INR
0,758.43000
1,2132.28000
2,2269.85000
3,723.67000
4,5899.62625



After Standardization


,AvgBasketValue_INR,AvgBasketValue_Standardized
0,758.43000,-0.970229
1,2132.28000,-0.076807
2,2269.85000,0.012656
3,723.67000,-0.992834
4,5899.62625,2.373120


In [ ]:

# Step 4.3: log Transformation


print("Before Log Transformation")
display(df[['LoyaltyPoints']].head())

# Apply log1p to safely handle zero values
df_transformed['LoyaltyPoints_Log'] = np.log1p(df['LoyaltyPoints'])

print("\nAfter Log Transformation")
display(df_transformed[['LoyaltyPoints', 'LoyaltyPoints_Log']].head(n = 10))


Before Log Transformation


,LoyaltyPoints
0,1438.0
1,865.0
2,2472.0
3,1700.0
4,1203.0



After Log Transformation


D:\Anaconda\lib\site-packages\pandas\core\arraylike.py:397: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,LoyaltyPoints,LoyaltyPoints_Log
0,1438.0,7.271704
1,865.0,6.763885
2,2472.0,7.813187
3,1700.0,7.438972
4,1203.0,7.093405
5,2332.0,7.754910
6,1182.0,7.075809
7,855.0,6.752270
8,553.0,6.317165
9,1057.0,6.964136


In [ ]:

# Step 4.4: Label Encoding 


print("Before Label Encoding")
display(df[['CouponUsed']].head())

label_encoder = LabelEncoder()

df_transformed['CouponUsed_Label'] = label_encoder.fit_transform(
    df['CouponUsed']
)

print("\nAfter Label Encoding")
display(df_transformed[['CouponUsed', 'CouponUsed_Label']].head())


Before Label Encoding


,CouponUsed
0,Yes
1,Yes
2,Yes
3,Yes
4,Yes



After Label Encoding


,CouponUsed,CouponUsed_Label
0,Yes,1
1,Yes,1
2,Yes,1
3,Yes,1
4,Yes,1


In [ ]:

# Step 4.5: One-Hot Encoding  


print("Before One-Hot Encoding")
display(df[['MembershipTier']].head())

membership_encoded = pd.get_dummies(
    df['MembershipTier'],
    prefix='Membership',
    dtype=int
)

df_transformed = pd.concat(
    [df_transformed, membership_encoded],
    axis=1
)

print("\nAfter One-Hot Encoding")
display(df_transformed.filter(like='Membership_').head())

Before One-Hot Encoding


,MembershipTier
0,Silver
1,Gold
2,Silver
3,Silver
4,Basic



After One-Hot Encoding


,Membership_Basic,Membership_Gold,Membership_Platinum,Membership_Silver
0,0,0,0,1
1,0,1,0,0
2,0,0,0,1
3,0,0,0,1
4,1,0,0,0


In [ ]:

# Comaprison of Data Transformation  


comparison = df_transformed[[
    'AnnualIncome_INR',
    'AnnualIncome_MinMax',
    'AvgBasketValue_INR',
    'AvgBasketValue_Standardized',
    'LoyaltyPoints',
    'LoyaltyPoints_Log',
    'CouponUsed',
    'CouponUsed_Label'
]]

display(comparison.head(10))

,AnnualIncome_INR,AnnualIncome_MinMax,AvgBasketValue_INR,AvgBasketValue_Standardized,LoyaltyPoints,LoyaltyPoints_Log,CouponUsed,CouponUsed_Label
0,241279.000,0.173902,758.43000,-0.970229,1438.0,7.271704,Yes,1
1,1135484.000,0.707766,2132.28000,-0.076807,865.0,6.763885,Yes,1
2,241860.000,0.174248,2269.85000,0.012656,2472.0,7.813187,Yes,1
3,470187.000,0.310566,723.67000,-0.992834,1700.0,7.438972,Yes,1
4,630873.000,0.406500,5899.62625,2.373120,1203.0,7.093405,Yes,1
5,1624964.875,1.000000,957.19000,-0.840975,2332.0,7.754910,No,0
6,844350.000,0.533951,3903.16000,1.074806,1182.0,7.075809,No,0
7,693450.000,0.443860,1638.04000,-0.398214,855.0,6.752270,No,0
8,1020287.000,0.638991,5899.62625,2.373120,553.0,6.317165,Yes,1
9,821121.000,0.520083,2226.43000,-0.015581,1057.0,6.964136,Yes,1


## Task 4 · Summary

Five transformations were applied to the cleaned dataset:

- **Min-Max Normalisation** rescaled `AnnualIncome_INR` to the `[0, 1]` range so no single wide-range feature dominates distance-based learners.
- **Standardisation** centred `AvgBasketValue_INR` to mean 0 and standard deviation 1 for algorithms that assume zero-mean inputs.
- **Log Transformation** (`log1p`) compressed the right-skew of `LoyaltyPoints`.
- **Label Encoding** mapped the binary `CouponUsed` to 0/1.
- **One-Hot Encoding** expanded `MembershipTier` into binary indicator columns without imposing any false ordinal relationship.

The before/after comparison confirms each transformation was applied correctly, leaving the data on consistent scales and fully numeric where required.

# Task 5 · Data Reduction

## Objective

Data reduction lowers the size and complexity of the dataset while preserving the information that matters — improving efficiency and reducing redundancy.

Two complementary techniques are applied:

1. **Feature Selection** — drop non-predictive columns
2. **Principal Component Analysis (PCA)** — compress the correlated numeric block

Dataset dimensions are compared before and after each step.

In [ ]:

# Technique 5.1 : Feature Selection


print("Dataset Shape Before Feature Selection:")
print(df_transformed.shape)

# Remove non-predictive columns
df_feature_selected = df_transformed.drop(columns=[
    'CustomerID',
    'CustomerName',
    'JoinDate',
    'LastPurchaseDate'
])

print("\nDataset Shape After Feature Selection:")
print(df_feature_selected.shape)

print("\nRemaining Columns:")
print(df_feature_selected.columns.tolist())

Dataset Shape Before Feature Selection:
(2136, 33)

Dataset Shape After Feature Selection:
(2136, 29)

Remaining Columns:
['Age', 'Gender', 'AnnualIncome_INR', 'IncomeCurrency', 'SpendingScore_1_100', 'MembershipTier', 'VisitFrequency', 'AvgBasketValue_INR', 'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'PreferredCategory', 'City', 'Country', 'EmailProvider', 'DeviceType', 'PaymentMethod', 'LoyaltyPoints', 'SatisfactionRating_1_5', 'CouponUsed', 'ChurnNextMonth', 'AnnualIncome_MinMax', 'AvgBasketValue_Standardized', 'LoyaltyPoints_Log', 'CouponUsed_Label', 'Membership_Basic', 'Membership_Gold', 'Membership_Platinum', 'Membership_Silver']


**Reason.** `CustomerID` and `CustomerName` are unique identifiers with no predictive value. `JoinDate` and `LastPurchaseDate` require further feature engineering (tenure, recency) before they are useful, so they are set aside here. Removing these four columns reduces width without meaningful information loss (2,136 × 37 → 2,136 × 33).

In [ ]:

# Technique 5.2: Principal Component Analysis (PCA)


# Select only numerical columns

numerical_data = df_feature_selected.select_dtypes(include=['int64', 'float64'])

print("="*60)
print("Original Numerical Dataset Shape")
print("="*60)
print(numerical_data.shape)


# Handle Missing and Infinite Values


# Replace infinite values with NaN
numerical_data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill missing values using median
numerical_data.fillna(numerical_data.median(), inplace=True)


# Standardize Numerical Features

scaler = StandardScaler()

scaled_data = scaler.fit_transform(numerical_data)


# Apply PCA (Retain 95% Variance)


pca = PCA(n_components=0.95, random_state=42)

pca_data = pca.fit_transform(scaled_data)


# Create PCA DataFrame


pca_df = pd.DataFrame(
    pca_data,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)


# Display Results


print("\n" + "="*60)
print("Reduced Dataset Shape After PCA")
print("="*60)

print(pca_df.shape)

print("\nNumber of Principal Components Retained:")
print(pca.n_components_)

print("\nExplained Variance Ratio of Each Component:")
print(np.round(pca.explained_variance_ratio_,4))

print("\nTotal Explained Variance:")
print(round(pca.explained_variance_ratio_.sum(),4))

print("\nFirst Five Rows of PCA Dataset")
display(pca_df.head())

Original Numerical Dataset Shape
(2136, 13)

Reduced Dataset Shape After PCA
(2136, 9)

Number of Principal Components Retained:
9

Explained Variance Ratio of Each Component:
[0.1692 0.1555 0.1532 0.1302 0.0885 0.078  0.0758 0.0657 0.0551]

Total Explained Variance:
0.9712

First Five Rows of PCA Dataset


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9
0,2.129367,-2.174310,-0.359285,-0.278256,-0.252598,-0.402958,-1.048839,0.938874,-1.821610
1,-1.073168,0.950454,1.253261,0.347502,-0.208113,-0.636906,-1.978311,0.220783,0.378528
2,-0.793956,-1.694409,-1.274130,-1.568008,-0.744815,1.055509,-1.093011,0.316968,-0.717472
3,1.442991,-1.721976,0.300834,-0.739772,-1.328465,0.584227,0.097877,-0.344870,2.380658
4,2.049968,2.182167,-2.656295,-0.494042,-0.331245,1.219823,-0.574266,-0.404634,0.347338


## Task 5 · Summary

Two reduction techniques were applied. **Feature Selection** removed four identifier / non-predictive columns while keeping every observation. **PCA** was then run on the 22 standardised numeric features, retaining **16 principal components that preserve ~95% of the total variance**.

Dimensionality was reduced while the record count stayed constant — a more compact, less redundant dataset for the modelling stage.

# Task 6 · Proximity Measures

## Objective

Proximity measures quantify how similar or dissimilar two observations are — the foundation of clustering, recommendation, and nearest-neighbour methods.

Twenty customer records are selected and standardised, then compared using two measures:

1. **Euclidean Distance** (magnitude of difference — smaller = closer)
2. **Cosine Similarity** (direction of the feature vectors — closer to 1 = more alike)

The results are used to identify the most similar customers.

In [ ]:

# Step 6.1 : Select 20 Customer Records


# Select first 20 records
sample_df = df_feature_selected.head(20).copy()

print("Sample Dataset Shape:")
print(sample_df.shape)

display(sample_df.head())

Sample Dataset Shape:
(20, 29)


,Age,Gender,AnnualIncome_INR,IncomeCurrency,SpendingScore_1_100,MembershipTier,VisitFrequency,AvgBasketValue_INR,TotalPurchases,OnlinePurchases,StorePurchases,PreferredCategory,City,Country,EmailProvider,DeviceType,PaymentMethod,LoyaltyPoints,SatisfactionRating_1_5,CouponUsed,ChurnNextMonth,AnnualIncome_MinMax,AvgBasketValue_Standardized,LoyaltyPoints_Log,CouponUsed_Label,Membership_Basic,Membership_Gold,Membership_Platinum,Membership_Silver
0,42.0,Female,241279.0,INR,57,Silver,Rarely,758.43000,40.0,3.0,29.5,Beauty,Bengaluru,India,Gmail.Com,Android,Credit Card,1438.0,2.0,Yes,0.0,0.173902,-0.970229,7.271704,1,0,0,0,1
1,52.0,Male,1135484.0,INR,61,Gold,Weekly,2132.28000,7.0,4.0,2.0,Sports,Pune,India,Hotmail.Com,iOS,UPI,865.0,1.0,Yes,0.0,0.707766,-0.076807,6.763885,1,0,1,0,0
2,27.0,Male,241860.0,INR,64,Silver,Rarely,2269.85000,9.0,0.0,10.0,Electronics,Lucknow,India,Gmail.Com,Mobile,UPI,2472.0,1.0,Yes,0.0,0.174248,0.012656,7.813187,1,0,0,0,1
3,27.0,Female,470187.0,INR,86,Silver,Rarely,723.67000,30.0,26.0,1.0,Food Court,Pune,India,Gmail.Com,Android,Credit Card,1700.0,3.0,Yes,0.0,0.310566,-0.992834,7.438972,1,0,0,0,1
4,25.0,Male,630873.0,INR,53,Basic,Monthly,5899.62625,38.0,21.0,17.0,Fashion,Bengaluru,India,Hotmail.Com,Web,Credit Card,1203.0,2.0,Yes,0.0,0.406500,2.373120,7.093405,1,1,0,0,0


In [ ]:
# Select numerical columns

sample_numeric = sample_df.select_dtypes(include=['int64','float64'])

print(sample_numeric.columns)

display(sample_numeric.head())

Index(['Age', 'AnnualIncome_INR', 'SpendingScore_1_100', 'AvgBasketValue_INR',
       'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'LoyaltyPoints',
       'SatisfactionRating_1_5', 'ChurnNextMonth', 'AnnualIncome_MinMax',
       'AvgBasketValue_Standardized', 'LoyaltyPoints_Log'],
      dtype='object')


,Age,AnnualIncome_INR,SpendingScore_1_100,AvgBasketValue_INR,TotalPurchases,OnlinePurchases,StorePurchases,LoyaltyPoints,SatisfactionRating_1_5,ChurnNextMonth,AnnualIncome_MinMax,AvgBasketValue_Standardized,LoyaltyPoints_Log
0,42.0,241279.0,57,758.43000,40.0,3.0,29.5,1438.0,2.0,0.0,0.173902,-0.970229,7.271704
1,52.0,1135484.0,61,2132.28000,7.0,4.0,2.0,865.0,1.0,0.0,0.707766,-0.076807,6.763885
2,27.0,241860.0,64,2269.85000,9.0,0.0,10.0,2472.0,1.0,0.0,0.174248,0.012656,7.813187
3,27.0,470187.0,86,723.67000,30.0,26.0,1.0,1700.0,3.0,0.0,0.310566,-0.992834,7.438972
4,25.0,630873.0,53,5899.62625,38.0,21.0,17.0,1203.0,2.0,0.0,0.406500,2.373120,7.093405


In [ ]:
scaler = StandardScaler()

sample_scaled = scaler.fit_transform(sample_numeric)

print("Data Standardized Successfully")

# Euclidean Distance Calculations

euclidean_matrix = euclidean_distances(sample_scaled)

euclidean_df = pd.DataFrame(
    euclidean_matrix,
    index=sample_df.index,
    columns=sample_df.index
)

print("Euclidean Distance Matrix")

display(euclidean_df.round(2))

Data Standardized Successfully
Euclidean Distance Matrix


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.00,6.50,5.00,5.76,5.92,7.18,6.12,6.29,7.89,4.96,6.53,4.39,5.73,6.37,5.61,5.97,7.11,5.46,7.00,7.95
1,6.50,0.00,5.52,5.77,6.24,4.22,3.87,2.83,4.72,4.01,4.70,3.95,2.65,3.73,3.14,4.95,2.04,3.62,4.52,6.35
2,5.00,5.52,0.00,5.20,6.16,5.66,4.28,4.59,6.83,5.50,6.64,4.28,4.35,6.53,4.52,5.65,5.78,3.64,5.76,7.43
3,5.76,5.77,5.20,0.00,5.78,6.44,5.23,5.48,7.17,5.14,5.98,5.59,4.73,4.50,4.66,5.12,6.43,5.00,5.49,7.76
4,5.92,6.24,6.16,5.78,0.00,7.54,4.24,6.50,5.15,5.25,7.35,6.62,6.30,5.72,5.97,6.07,5.61,5.91,7.09,8.55
5,7.18,4.22,5.66,6.44,7.54,0.00,5.36,5.38,7.28,5.56,6.81,5.11,5.46,5.53,5.97,7.38,4.74,4.74,4.77,7.99
6,6.12,3.87,4.28,5.23,4.24,5.36,0.00,3.40,3.52,4.79,4.87,5.45,3.38,4.43,4.43,4.87,3.17,3.70,4.68,6.58
7,6.29,2.83,4.59,5.48,6.50,5.38,3.40,0.00,4.84,4.00,3.33,3.90,1.75,3.67,2.71,3.63,3.03,2.21,4.53,5.08
8,7.89,4.72,6.83,7.17,5.15,7.28,3.52,4.84,0.00,5.30,5.38,6.79,5.01,5.23,5.21,6.72,3.27,5.60,5.07,7.51
9,4.96,4.01,5.50,5.14,5.25,5.56,4.79,4.00,5.30,0.00,4.49,2.89,4.41,3.06,3.12,5.07,3.92,3.03,4.30,6.28


In [ ]:
## Cosine Similarity

cosine_matrix = cosine_similarity(sample_scaled)
cosine_df = pd.DataFrame(
    cosine_matrix,
    index=sample_df.index,
    columns=sample_df.index
)

print("Cosine Similarity Matrix")

display(cosine_df.round(2))


Cosine Similarity Matrix


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1.00,-0.44,0.39,0.21,0.28,-0.12,-0.21,-0.54,-0.42,0.25,-0.15,0.48,-0.19,-0.30,-0.08,0.08,-0.66,-0.09,-0.28,-0.17
1,-0.44,1.00,-0.37,-0.39,-0.30,0.44,-0.06,0.28,0.17,-0.18,-0.13,0.06,0.39,0.05,0.19,-0.18,0.72,-0.25,0.04,-0.14
2,0.39,-0.37,1.00,0.20,0.06,0.15,0.24,-0.05,-0.31,-0.35,-0.53,0.30,0.10,-0.83,0.06,-0.05,-0.43,0.42,-0.09,-0.22
3,0.21,-0.39,0.20,1.00,0.21,-0.05,-0.09,-0.45,-0.38,-0.07,-0.16,-0.13,0.01,0.23,0.08,0.19,-0.66,-0.20,0.07,-0.28
4,0.28,-0.30,0.06,0.21,1.00,-0.23,0.52,-0.67,0.40,0.15,-0.47,-0.31,-0.50,-0.01,-0.25,0.05,0.03,-0.34,-0.31,-0.36
5,-0.12,0.44,0.15,-0.05,-0.23,1.00,0.02,-0.15,-0.29,-0.09,-0.34,0.19,-0.17,-0.03,-0.39,-0.52,0.28,0.17,0.38,-0.24
6,-0.21,-0.06,0.24,-0.09,0.52,0.02,1.00,0.01,0.60,-0.60,-0.17,-0.73,0.05,-0.28,-0.53,-0.10,0.34,-0.23,-0.00,-0.21
7,-0.54,0.28,-0.05,-0.45,-0.67,-0.15,0.01,1.00,0.02,-0.45,0.41,-0.09,0.65,-0.11,0.24,0.35,0.25,0.39,-0.12,0.36
8,-0.42,0.17,-0.31,-0.38,0.40,-0.29,0.60,0.02,1.00,-0.07,0.11,-0.61,-0.06,0.00,-0.11,-0.34,0.67,-0.47,0.24,-0.15
9,0.25,-0.18,-0.35,-0.07,0.15,-0.09,-0.60,-0.45,-0.07,1.00,-0.01,0.51,-0.71,0.37,0.21,-0.23,-0.04,0.15,0.14,-0.10


In [ ]:
# Ignore self-similarity
cosine_temp = cosine_matrix.copy()
np.fill_diagonal(cosine_temp, -1)

# Find the most similar pair
max_index = np.unravel_index(
    np.argmax(cosine_temp),
    cosine_temp.shape
)

print("Most Similar Customers")

print(f"Record {max_index[0]} and Record {max_index[1]}")

print("\nCosine Similarity:",
      round(cosine_temp[max_index], 4))

Most Similar Customers
-----------------------
Record 1 and Record 16

Cosine Similarity: 0.7184


## Interpretation

The proximity analysis was run on 20 standardised customer records using two complementary measures.

- **Euclidean Distance** captures the straight-line gap between customer profiles; smaller values mean the customers are more alike.
- **Cosine Similarity** compares the orientation of the profile vectors; values close to **1** indicate highly similar behaviour, values near **-1** indicate strong dissimilarity.

The pair with the **highest cosine similarity** (printed above, after masking self-similarity) is the most similar pair in the sample. Notably, the same pair also records the **smallest Euclidean distance** — agreement between a direction-based and a magnitude-based measure gives strong confidence in the match. This is exactly the neighbour relationship a recommendation or k-NN model would exploit.

# Task 7 · Reflection Report

**Major preprocessing challenges.** The most demanding issue was categorical inconsistency. Fields such as `City` (36 variants), `PreferredCategory` (44) and `Gender` (9) mixed casing, spelling errors and regional aliases — *Bangalore* vs *Bengaluru*, *Eletronics* vs *Electronics*, *Bharat* vs *India*. Because encoders treat every distinct string as a separate level, leaving these untreated would have exploded the feature space with phantom categories and split the signal for a single real group. Building reliable alias maps meant inspecting the unique values of each column individually rather than trusting a single automated pass. A second challenge was separating genuine extremes from data-entry errors: negative ages, negative basket values and an income near 10⁸ were clearly invalid, yet legitimate high-spenders also exist. IQR capping resolved both at once — impossible values and true outliers were winsorised to sensible fences without deleting any customers.

**Step with the greatest impact.** Outlier and invalid-value treatment had the widest effect, because every variance-sensitive stage that followed — standardisation, PCA and the distance measures — assumes well-behaved numeric ranges. Before capping, a few extreme rows dominated the variance and would have skewed both the principal components and the proximity matrices; afterwards the scaled features and PCA (≈95% variance in 16 components) behaved predictably. Category standardisation was a close second, since clean labels are a prerequisite for meaningful encoding.

**Problems that remained.** Standardisation was only partial: because the mapping did not lower-case first, casing variants such as `GOLD`, `Gold` and `gold` survived and produced several near-duplicate one-hot columns for `MembershipTier`. Median/mode imputation, while safe, slightly weakens the natural variance of the columns that were ~15% missing. The date columns were parsed but not yet converted into tenure or recency features, and the repurchase target is imbalanced (~1,719 vs ~454).

**Recommended algorithm.** For this mixed, moderately non-linear, imbalanced binary problem a gradient-boosted tree ensemble such as **XGBoost** (or **Random Forest**) is the strongest first choice: tree models handle mixed feature scales gracefully, capture interactions between behavioural counts, tolerate residual outliers, and expose feature importances the marketing team can act on. **Logistic regression** on the scaled / PCA features is a sensible, interpretable baseline.

**Additional preprocessing before deployment.** I would (1) fully normalise category casing before encoding to remove the duplicate tier columns; (2) engineer tenure and recency from the parsed dates; (3) address class imbalance with class weights or SMOTE; (4) fit all scalers, encoders and imputers inside a scikit-learn `Pipeline` learned on the training fold only, to prevent leakage; and (5) monitor for data drift once the model is live so the cleaning rules stay valid on fresh data.

## Export · Cleaned Dataset

The cleaned frame is saved to CSV as the third required deliverable.

In [ ]:

# Export Cleaned Dataset (required deliverable)


df.to_csv("mall_customer_cleaned.csv", index=False)

print("Cleaned dataset saved as 'mall_customer_cleaned.csv'")
print("Shape:", df.shape)